# Direttore modular-monolith warehouse

Warehouse and Orders are separate bounded contexts with their own registries and Unit of Work types. They share one resource boundary, while the typed implementation lives in [`src/modular_monolith`](../../src/modular_monolith). See [`docs/modular-monolith.md`](../../docs/modular-monolith.md) for the framework configuration reference.

In [1]:
from pathlib import Path
import sys

project_src = next(
    parent / 'src'
    for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / 'src/modular_monolith').exists()
)
sys.path.insert(0, str(project_src.resolve()))

from modular_monolith.bootstrap.application import build_application
from modular_monolith.bootstrap.contexts import contexts
from modular_monolith.contexts.orders.application.use_cases import PlaceOrderCommand
from modular_monolith.contexts.warehouse.application.errors import InsufficientStockError
from modular_monolith.contexts.warehouse.application.use_cases import (
    GetStockCommand, ReceiveStockCommand, RegisterProductCommand,
)
from modular_monolith.shared.lifecycle import RequestInput

## Two contexts, one composition root

Each `context.py` imports its registration modules and exports a `ModularMonolithDirettoreContext`. Bootstrap supplies both contexts, a shared holder factory, and a coordinator that creates the concrete context UoWs.

In [2]:
example = build_application()
[(context.use_case_registry.source_name, context.use_case_root_uow_type.__name__) for context in contexts]

[('warehouse', 'InMemoryWarehouseUnitOfWork'),
 ('orders', 'InMemoryOrdersUnitOfWork')]

## Prepare Warehouse

The receipt client is a reusable port adapter injected into `ReceiveStockHandler`. Warehouse repositories are assembled inside `InMemoryWarehouseUnitOfWork`, not in the dependency container.

In [3]:
request = RequestInput(actor_id='notebook', correlation_id='modular-100')
await example.application.handle(
    RegisterProductCommand('P-100', 'Keyboard'), input=request
)
await example.application.handle(ReceiveStockCommand('P-100', 10))
example.receipt_client.calls

[('P-100', 10)]

## Cross-context invocation reuses the session

`PlaceOrderHandler` depends on the Orders-owned `WarehouseContextClient` port. Its execution-scoped `InProcessWarehouseContextClient` adapter calls the active modular runtime, which routes `ReserveStockCommand` to the Warehouse UoW. The reservation, order write, and event audit below all have one session ID.

In [4]:
access_start = len(example.database.access_log)
order = await example.application.handle(
    PlaceOrderCommand('O-100', 'P-100', 3), input=request
)
operation_accesses = example.database.access_log[access_start:]
order, operation_accesses, {session_id for _, session_id in operation_accesses}

(OrderSnapshot(order_id='O-100', product_id='P-100', quantity=3, status='placed'),
 [('warehouse.products.reserve', 3),
  ('warehouse.products.get', 3),
  ('orders.orders.add', 3),
  ('orders.audits.record', 3)],
 {3})

## Failure rolls back both context changes

The nested Warehouse reservation fails before Orders stores the order. The one shared resource is rolled back and closed.

In [5]:
try:
    await example.application.handle(PlaceOrderCommand('O-101', 'P-100', 8))
except InsufficientStockError as error:
    print(f'Expected failure: {error}')

stock = await example.application.handle(GetStockCommand('P-100'))
stock, 'O-101' in example.database.orders, example.database.transaction_log[-4:]

Expected failure: requested=8, available=7


(StockBalance(product_id='P-100', quantity=7),
 False,
 [('rollback', 4), ('close', 4), ('rollback', 5), ('close', 5)])

## Warehouse saga context

Warehouse declares a typed `WarehouseSagaContext` in `application/architecture.py`. A receipt recorded with a saga ID can later be reversed through the same Warehouse UoW.

In [6]:
await example.application.handle(RegisterProductCommand('P-200', 'Mouse'))
await example.application.handle(
    ReceiveStockCommand('P-200', 2), saga_id='receipt-200'
)
before = await example.application.handle(GetStockCommand('P-200'))
await example.application.compensate_saga('receipt-200')
after = await example.application.handle(GetStockCommand('P-200'))
before, after

(StockBalance(product_id='P-200', quantity=2),
 StockBalance(product_id='P-200', quantity=0))

## Inspect shared effects

Audits from both event registries land in the same committed resource. Restart the kernel and run all cells to reset the example.

In [7]:
example.database.products, example.database.orders, example.database.audits, example.application.slot_provider_stats()

({'P-100': {'product_id': 'P-100', 'name': 'Keyboard', 'quantity': 7},
  'P-200': {'product_id': 'P-200', 'name': 'Mouse', 'quantity': 0}},
 {'O-100': {'order_id': 'O-100',
   'product_id': 'P-100',
   'quantity': 3,
   'status': 'placed'}},
 [{'context': 'warehouse',
   'kind': 'stock_received',
   'product_id': 'P-100',
   'quantity': 10,
   'new_balance': 10},
  {'context': 'orders',
   'kind': 'order_placed',
   'order_id': 'O-100',
   'product_id': 'P-100',
   'quantity': 3},
  {'context': 'warehouse',
   'kind': 'stock_received',
   'product_id': 'P-200',
   'quantity': 2,
   'new_balance': 2}],
 ExecutionSlotProviderStats(total_slots=1, free_slots=1, acquired_slots=0, max_slots=2))